# Generate Restored SPFC full with unclipped VFA on T2I-CompBench indexes 50 to 99 (Kaggle)

Runs only `spfc_vfa_unclipped` for the end-exclusive manifest slice `50:100`.

Outputs retain the standard merge-compatible path under `/kaggle/working/t2i_compbench_seed13/runs/t2i_compbench/spfc_vfa_unclipped`.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Soobiwan/aim-flow.git'

%cd /kaggle/working
!rm -rf /kaggle/working/aim-flow
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
from pathlib import Path
import json
import os
import shutil
import shlex
import subprocess
import sys
import time

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("DIFFUSERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SEED = 13
RUN_SLUG = "t2i_compbench_seed13"
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / RUN_SLUG
RUN_ROOT = OUTPUT_ROOT / "runs"
MANIFEST_REL = Path("configs/t2i_compbench_100_seed13.json")
DECOMP_REL = Path("configs/t2i_compbench_100_seed13_spfc.json")
CONFIG_REL = Path("configs/sd3_medium_kaggle_vfa_unclipped.yaml")
RECTIFIED_REPO_DIR = WORK_ROOT / "Rectified-CFGpp"
INSTALL_DEPS = True
GPU_SELECTION = "0"
SHARD_START = 50
SHARD_END = 100
EXPECTED_SHARD_COUNT = SHARD_END - SHARD_START
SHARD_LABEL = "050-099"

# If your repo is attached with a different Kaggle dataset slug, this auto-discovers it under /kaggle/input.
def has_aim_flow_repo(path: Path) -> bool:
    return (path / "src" / "aim_flow").exists() and (path / "scripts" / "bench_generate.py").exists()


def find_aim_flow_repo() -> Path | None:
    cwd = Path.cwd()
    if has_aim_flow_repo(cwd):
        return cwd
    dest = WORK_ROOT / "aim-flow"
    if has_aim_flow_repo(dest):
        return dest
    input_root = Path("/kaggle/input")
    candidates = [input_root / "aim-flow", input_root / "aim-flow" / "aim-flow"]
    if input_root.exists():
        for child in sorted(input_root.glob("*")):
            candidates.extend([child, child / "aim-flow"])
    for candidate in candidates:
        if has_aim_flow_repo(candidate):
            return candidate
    return None


def ensure_working_repo() -> Path:
    source = find_aim_flow_repo()
    if source is None:
        raise FileNotFoundError(
            "Could not find aim-flow. Attach the repo as a Kaggle dataset, clone it into /kaggle/working, "
            "or run this notebook from the repo root."
        )
    dest = WORK_ROOT / "aim-flow" if Path("/kaggle").exists() else source
    if source.resolve() != dest.resolve():
        shutil.copytree(source, dest, dirs_exist_ok=True)
        return dest
    return source


def run_args(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", shlex.join([str(arg) for arg in args]))
    started = time.time()
    result = subprocess.run([str(arg) for arg in args], cwd=str(cwd) if cwd else None, text=True)
    print(f"elapsed: {(time.time() - started) / 60:.2f} min")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def normalize_gpu_selection(value: str) -> str:
    selection = ",".join(part.strip() for part in str(value).split(",") if part.strip())
    if not selection:
        raise ValueError("GPU_SELECTION must be a non-empty string like '0' or '0,1'.")
    if any(not part.isdigit() for part in selection.split(",")):
        raise ValueError(f"GPU_SELECTION must contain only GPU indices, got: {value!r}")
    return selection


REPO_DIR = ensure_working_repo()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
GPU_SELECTION = normalize_gpu_selection(GPU_SELECTION)
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_SELECTION
print("repo:", REPO_DIR)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("outputs:", OUTPUT_ROOT)
print("config:", REPO_DIR / CONFIG_REL)

if INSTALL_DEPS:
    run_args([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=REPO_DIR)
    run_args([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=REPO_DIR)

In [ ]:
METHOD_LABEL = "spfc_vfa_unclipped"
METHOD_TITLE = "Restored SPFC full with unclipped VFA"
SPFC_VARIANT = None
GUIDANCE_SCALE = 4.5
print(f"Generating {METHOD_TITLE} shard {SHARD_LABEL}: indexes {SHARD_START} to {SHARD_END - 1}")

In [ ]:
import os
from IPython.display import Markdown, display

MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
HF_SECRET_NAMES = ('Huggingface', 'HF_TOKEN', 'HUGGINGFACE_TOKEN')

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in HF_SECRET_NAMES:
            try:
                token = secrets.get_secret(secret_name)
            except Exception:
                token = None
            if token:
                os.environ['HF_TOKEN'] = token
                break
    except Exception:
        pass

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not token:
    raise RuntimeError(
        'Missing Hugging Face token. In Kaggle, add a secret named Huggingface or HF_TOKEN, '
        'turn it on for this notebook, and make sure that Hugging Face account has accepted the SD3 Medium license.'
    )

try:
    from huggingface_hub import HfApi
    HfApi().model_info(MODEL_ID, token=token)
except Exception as exc:
    raise RuntimeError(
        f'HF_TOKEN is set, but access check for {MODEL_ID} failed. '
        'Confirm the Kaggle secret is enabled and the token account has accepted the gated model license.'
    ) from exc

display(Markdown('Hugging Face token is configured and can access SD3 Medium.'))


run_args(["nvidia-smi"], check=False)

In [ ]:
# If Kaggle gives a P100 with an incompatible Torch build, uncomment these and restart the runtime.
!pip uninstall -q -y torch torchvision torchaudio
!pip install -q --no-cache-dir --force-reinstall torch==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps -e .


In [ ]:
manifest_path = REPO_DIR / MANIFEST_REL
decomp_path = REPO_DIR / DECOMP_REL
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)
if not decomp_path.exists():
    raise FileNotFoundError(decomp_path)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)
with decomp_path.open("r", encoding="utf-8") as f:
    decompositions = json.load(f)

sample_count = len(manifest["samples"])
decomposition_count = len(decompositions["items"])
expected_ids = [sample["id"] for sample in manifest["samples"][SHARD_START:SHARD_END]]
print("manifest:", manifest_path)
print("decompositions:", decomp_path)
print("benchmark:", manifest["benchmark"])
print("full samples:", sample_count)
print("shard samples:", len(expected_ids))
assert sample_count == 100, f"Expected 100 T2I-CompBench samples, found {sample_count}."
assert decomposition_count == 100, f"Expected 100 SPFC decompositions, found {decomposition_count}."
assert len(expected_ids) == EXPECTED_SHARD_COUNT == 50

from aim_flow.eval_bench.generation import apply_spfc_variant, load_bench_config

full_config = load_bench_config(config_path=REPO_DIR / CONFIG_REL, seed=SEED, guidance_scale=GUIDANCE_SCALE)
primitive_flow = full_config.primitive_flow
assert primitive_flow.uniform_condition_weights is False
assert primitive_flow.use_consensus_gating is True
assert primitive_flow.use_target_consistency_gating is True
assert primitive_flow.source_weight == 0.7
assert primitive_flow.target_weight == 1.2
assert primitive_flow.velocity_clip_ratio == 1000000.0
print("validated restored SPFC full settings with unclipped VFA")

In [ ]:
cmd = [
    sys.executable,
    "scripts/bench_generate.py",
    "--manifest",
    REPO_DIR / MANIFEST_REL,
    "--run-root",
    RUN_ROOT,
    "--methods",
    "spfc",
    "--decompositions",
    REPO_DIR / DECOMP_REL,
    "--config",
    REPO_DIR / CONFIG_REL,
    "--seed",
    str(SEED),
    "--guidance-scale",
    str(GUIDANCE_SCALE),
    "--sample-start",
    str(SHARD_START),
    "--sample-end",
    str(SHARD_END),
    "--spfc-method-label",
    METHOD_LABEL,
    "--skip-existing",
]
run_args(cmd, cwd=REPO_DIR)

In [ ]:
method_dir = RUN_ROOT / "t2i_compbench" / METHOD_LABEL
index_path = method_dir / "index.json"
if not index_path.exists():
    raise FileNotFoundError(f"Missing generation index: {index_path}")
with index_path.open("r", encoding="utf-8") as f:
    index = json.load(f)
output_ids = [item["sample_id"] for item in index["outputs"]]
assert output_ids == expected_ids, f"Unexpected output IDs for {METHOD_LABEL}"
print(f"{METHOD_LABEL}: {len(output_ids)} outputs at {method_dir}")

readme = OUTPUT_ROOT / f"README_{METHOD_LABEL}_shard_{SHARD_LABEL}.txt"
readme.write_text(
    "T2I-CompBench SPFC shard\n"
    f"method: {METHOD_LABEL}\n"
    f"variant: {SPFC_VARIANT or 'full'}\n"
    f"indexes: {SHARD_START} to {SHARD_END - 1}\n"
    f"slice: {SHARD_START}:{SHARD_END}\n"
    f"seed: {SEED}\n"
    f"guidance_scale: {GUIDANCE_SCALE}\n"
    f"method_dir: {method_dir}\n",
    encoding="utf-8",
)
print("index:", index_path)
print("readme:", readme)
print("Kaggle output root:", OUTPUT_ROOT)
print("Merge this notebook output with the matching method's other shard before the comparison run.")